In [2]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import h5py

from glob import glob
from shapely import contains_xy
from spectral.io import envi
import rasterio
from pyproj import Transformer

In [3]:
# fp = r"C:\Users\carroll\Documents\sbgplant\data\CRBU2018_AOP_Crowns.geojson"
# plots = gpd.read_file(fp)
# plots = plots.rename(columns={'SiteCode': 'plot_name'})
# plots['campaign_name'] = 'East River 2018'
# plots['sensor_name'] = 'NEON AIS 1'
# plots = plots[['plot_name', 'campaign_name', 'sensor_name', 'geometry']]
# plots = plots.to_crs(epsg=32613)
# poly = plots.geometry.union_all()

In [33]:
fp_v1 = r"C:\Users\carroll\Downloads\2018_FullSite_D13_2018_CRBU_1_L1_Spectrometer_RadianceH5_2018061314_NEON_D13_CRBU_DP1_20180613_175553_radiance.h5"
fp_v2 = r"C:\Users\carroll\Downloads\2018_FullSite_D13_2018_CRBU_1_L1_Spectrometer_RadianceH5_2018061314_NEON_D13_CRBU_DP1_L072-1_20180613_radiance_v2.h5"

In [34]:
with h5py.File(fp_v1, "r") as f:
    time = f[f'/CRBU/Radiance'].attrs['Acquisition_Time']
acquisition_date = time.split(',')[0]
acquisition_start_time = time.split(',')[1].replace('[Computer Time in sec]', '').strip()
fid = f'NIS01_{acquisition_date.replace("-", "")}_{acquisition_start_time}'
fid

'NIS01_20180613_175553'

In [12]:
# get rows and cols of px within plot polys
# with h5py.File(fp_v1, "r") as f:
#     igm = f['/CRBU/Radiance/Metadata/Ancillary_Rasters/IGM_Data'][:]
    
# x = igm[:, :, 0]
# y = igm[:, :, 1]
# inside = contains_xy(poly, x, y)
# rows, cols = np.where(inside)
# print(len(rows), 'px inside polys')

231 px inside polys


In [36]:
with h5py.File(fp_v1, "r") as f:
    igm = f['/CRBU/Radiance/Metadata/Ancillary_Rasters/IGM_Data'][:]
    
# lon = x[rows, cols]
# lat = y[rows, cols]
# elev = igm[rows, cols, -1]

elev = igm[:,:,-1]
del igm

print(elev.shape)

(1484, 5960)


In [37]:
# # obs
# with h5py.File(fp, "r") as f:
#     obs = f['/CRBU/Radiance/Metadata/Ancillary_Rasters/OBS_Data'][:]
# obs = obs[rows, cols, :]
# path_length = obs[:, 0]
# to_sensor_azimuth = obs[:, 1]
# to_sensor_zenith = obs[:, 2]
# to_sun_azimuth = obs[:, 3]
# to_sun_zenith = obs[:, 4]
# solar_phase = obs[:, 5]
# slope = obs[:, 6]
# aspect = obs[:, 7]
# cosine_i = obs[:, 8]
# utc_time = obs[:, 9]
# del obs

In [15]:
# # need to be sure the glt matches the file. So realistically should be using the updated v2 radiance files
# with h5py.File(fp, "r") as f:
#     def print_structure(name, obj):
#         print(name, type(obj))

#     f.visititems(print_structure)

In [ ]:
# old h5 file does not have glt in it. Does the new one?

In [37]:
# glt (v1)
glt = envi.open(r"C:\Users\carroll\Downloads\NIS01_20180613_175553\NIS01_20180613_175553_rdn_ort_glt.hdr").open_memmap()

glt_col = glt[:,:, 0]
glt_row = glt[:,:, 1]

glt_col.shape, glt_row.shape


((1484, 5960), (1484, 5960))

In [19]:
# glt (v2)
with h5py.File(fp_v2, "r") as f:
    glt = f[f'/{domain}/Radiance/Metadata/Ancillary_Rasters/GLT_Data'][:]
# glt_col = glt[rows, cols, 0]
# glt_row = glt[rows, cols, 1]

glt_col = glt[:,:, 0]
glt_row = glt[:,:, 1]

glt_col.shape, glt_row.shape

((1484, 5960), (1484, 5960))

In [38]:
df = pd.DataFrame({
    'row': np.abs(glt_row.ravel()),
    'col': np.abs(glt_col.ravel()),
    'elev': elev.ravel()
})

df = df[(df.row > 0) & (df.col > 0)]

print(df.shape)

(5386864, 3)


In [39]:
stats = (
    df.groupby(['row', 'col'])['elev']
      .agg(
          n='size',
          elev_min='min',
          elev_max='max',
          elev_std='std'
      )
      .reset_index()
)

stats['elev_range'] = stats['elev_max'] - stats['elev_min']

stats = stats[stats.elev_range > 0]

In [40]:
stats

,row,col,n,elev_min,elev_max,elev_std,elev_range
9,1,12,2,-9999.000000,2931.428223,9143.193359,12930.427734
50,1,59,2,-9999.000000,2917.194824,9133.128906,12916.195312
190,1,208,2,2879.298828,2879.299561,0.000518,0.000732
336,1,360,2,2826.729980,2826.929932,0.141387,0.199951
371,1,396,2,2815.255615,2815.312500,0.040224,0.056885
...,...,...,...,...,...,...,...
4454486,11640,459,2,-9999.000000,2972.011719,9171.890625,12971.011719
4454510,11640,489,3,2985.063965,2985.645508,0.335754,0.581543
4454567,11640,554,2,-9999.000000,2955.140625,9159.960938,12954.140625
4454590,11640,581,2,-9999.000000,2950.990723,9157.026367,12949.990234


In [40]:
df = pd.DataFrame({
    'granule_id': fid,
    'x': x[rows, cols],
    'y': y[rows, cols],
    'glt_row': glt_row,
    'glt_column': glt_col,
    'lon': lon,
    'lat': lat,
    'elevation': elev,
    'path_length': path_length,
    'to_sensor_azimuth': to_sensor_azimuth,
    'to_sensor_zenith': to_sensor_zenith,
    'to_sun_azimuth': to_sun_azimuth,
    'to_sun_zenith': to_sun_zenith,
    'solar_phase': solar_phase,
    'slope': slope,
    'aspect': aspect,
    'cosine_i': cosine_i,
    'utc_time': utc_time,
})

In [43]:
dup_rows = df[df.duplicated(
    subset=['granule_id', 'glt_row', 'glt_column'],
    keep=False
)]
dup_rows

,granule_id,x,y,glt_row,glt_column,lon,lat,elevation,path_length,to_sensor_azimuth,to_sensor_zenith,to_sun_azimuth,to_sun_zenith,solar_phase,slope,aspect,cosine_i,utc_time
12,NIS01_20180613_175553,330938.03125,4309930.0,-3320.0,-172.0,330938.03125,4309930.0,2760.376221,1702.284668,168.130997,6.449301,130.990158,21.793406,17.080112,20.250988,48.769489,0.888526,17.951296
14,NIS01_20180613_175553,330938.03125,4309930.0,-3320.0,-172.0,330938.03125,4309930.0,2760.376221,1702.284668,168.130997,6.449301,130.990158,21.793406,17.080112,20.250988,48.769489,0.888526,17.951296


In [ ]:
# the problem actually just goes away if we use the v2 radiance data?
# as a check, are there duplicates in the 2025 data? Are their data the same